In [ ]:
%%capture
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
# import seaborn as sns
import scipy.stats as stats
from datetime import datetime
from django_pandas.io import read_frame
from pathlib import Path


from dj_notebook import activate

# pd.options.mode.copy_on_write = True
# pd.options.mode.chained_assignment = "raise"
env_file = os.environ["INTECOMM_ENV"]
documents_folder = os.environ["INTECOMM_DOCUMENTS_FOLDER"]
plus = activate(dotenv_file=env_file)

report_folder = Path(documents_folder)

In [ ]:
from edc_model_to_dataframe import read_frame_edc
from edc_pdutils.dataframes import get_subject_visit
from edc_appointment.models import Appointment
from intecomm_subject.models import SubjectVisit, SubjectVisitMissed, HivReview, DrugRefillHiv

In [ ]:
from intecomm_analytics.dataframes import get_df_main_1858

df_main = get_df_main_1858(None)

In [ ]:
df_appt= read_frame(Appointment.objects.all())
df_appt[df_appt.visit_code_sequence==0].appt_status.value_counts()


In [ ]:
df_visit= read_frame_edc(SubjectVisit.objects.all())
df_visit.rename(columns={'id':'subject_visit_id'}, inplace=True)
df_missed= read_frame_edc(SubjectVisitMissed.objects.all())
df_missed.rename(columns={"report_datetime": "missed_visit_report_datetime"}, inplace=True)
df_visit = df_visit.merge(df_missed[["subject_visit_id", "missed_visit_report_datetime"]], on="subject_visit_id", how="left")

In [ ]:
df_visit = df_visit.merge(df_main[["subject_identifier", "assignment", "htn", "hiv", "hiv_only", "dm"]], on="subject_identifier", how="left")


In [ ]:
df_visit[(df_visit.assignment=="a") & (df_visit.visit_code_sequence==0) & (df_visit.missed_visit_report_datetime.notna())].visit_code.value_counts()

In [ ]:
# df_visit[(df_visit.assignment=="b") & (df_visit.visit_code_sequence==0) & (df_visit.missed_visit_report_datetime.notna())].visit_code.value_counts()
df_visit[(df_visit.assignment=="b") & (df_visit.visit_code_sequence==0) & (df_visit.missed_visit_report_datetime.notna())][["subject_identifier", "visit_code", "hiv","hiv_only", "dm", "htn"]]


In [ ]:
# hiv expected every 3 months or base on rx_days from hiv_review
# baseline, endline and 2-3 visit between
df_visit[(df_visit.hiv_only==1) & (df_visit.assignment=="b") & (df_visit.visit_code_sequence==0) & (df_visit.missed_visit_report_datetime.isna())][["subject_identifier", "visit_code", "hiv", "dm", "htn"]].groupby(by=["subject_identifier"]).size()

In [ ]:
# who is on a 30 day prescription?
df_hiv_refill = read_frame_edc(DrugRefillHiv.objects.all())
df_hiv_refill[df_hiv_refill.subject_identifier=="107-209-0010-8"].sort_values(by=["visit_code"])[["rx_days", "visit_code", "visit_datetime"]]

In [ ]:
from dateutil import parser

data = {"days": [30,30,30,30,30,30,30,30,30,30],
"visit_code": ["1020",
"1030",
"1040",
"1050",
"1060",
"1070",
"1080",
"1090",
"1100",
"1110"
],
        "visit_datetime": [parser.parse("2023-04-17 05:00:00"),
parser.parse("2023-05-15 05:00:00"),
parser.parse("2023-06-13 06:46:01"),
parser.parse("2023-07-13 06:22:46"),
parser.parse("2023-08-10 09:24:46"),
parser.parse("2023-09-12 11:13:38"),
parser.parse("2023-10-10 07:28:50"),
parser.parse("2023-11-08 06:00:00"),
parser.parse("2023-12-11 06:00:00"),
parser.parse("2024-02-13 07:29:41")
]}
df = pd.DataFrame(data)

In [ ]:
df

In [ ]:
df["next_visit_datetime"] = df["visit_datetime"].shift(-1)
df

In [ ]:
from datetime import timedelta

df["interval_check"] = (df["visit_datetime"] + pd.to_timedelta(df["days"], unit='d') + timedelta(days=3)) >= df["next_visit_datetime"]
df["interval_days"] = df["next_visit_datetime"] - (df["visit_datetime"] + pd.to_timedelta(df["days"], unit='d'))
df["interval_days"] = df.apply(lambda row: timedelta(days=0) if row.interval_days <= timedelta(days=0) else row.interval_days, axis=1)

df

In [ ]:
df_hiv_refill = read_frame_edc(DrugRefillHiv.objects.all())
# df_hiv_refill[df_hiv_refill.subject_identifier=="107-209-0010-8"].sort_values(by=["visit_code"])[["subject_identifier", "rx_days", "visit_code", "visit_datetime"]]
df = df_hiv_refill.copy()
df = df.merge(df_main[["subject_identifier", "assignment"]], on="subject_identifier", how="left")
df.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
df.reset_index(drop=True, inplace=True)


df["next_visit_datetime"] = df.groupby("subject_identifier")["visit_datetime"].shift(-1)
df["interval_check"] = (df["visit_datetime"] + pd.to_timedelta(df["rx_days"], unit='d') + timedelta(days=3)) >= df["next_visit_datetime"]
df["interval_days"] = df["next_visit_datetime"] - (df["visit_datetime"] + pd.to_timedelta(df["rx_days"], unit='d'))
df["interval_days"] = df.apply(lambda row: timedelta(days=0) if row.interval_days <= timedelta(days=0) else row.interval_days, axis=1)



In [ ]:
df.sort_values(by=["subject_identifier", "visit_datetime"], inplace=True)
# df[["subject_identifier", "visit_datetime", "visit_code", "next_visit_datetime", "interval_days", "interval_check", "rx_days"]]

In [ ]:
df

In [ ]:
# df[["subject_identifier", "visit_datetime", "assignment", "interval_days"]].to_csv("~/intervals.csv", index=False)
df = df[["subject_identifier", "visit_datetime", "assignment", "interval_days"]].copy()
df.reset_index(drop=True, inplace=True)

In [ ]:
df["interval_days"] = df["interval_days"].apply(lambda x: x.days)

In [ ]:
print(df.head())

In [ ]:

df["interval_days"].describe()

In [ ]:
df_a = df[df["assignment"]=="a"].groupby("subject_identifier")["interval_days"].sum()
df_b = df[df["assignment"]=="b"].groupby("subject_identifier")["interval_days"].sum()
#df_ab = pd.concat([df_a, df_b])
#df_ab


In [ ]:
df_a.describe()

In [ ]:
df_b.describe()

In [ ]:
print("Total interval days per person in group A:")
print(df_a)
print("\nTotal interval days per person in group B:")
print(df_b)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro

In [ ]:
# Plot histograms and Q-Q plots to visually inspect the distribution
plt.figure(figsize=(12, 6))

plt.subplot(2, 2, 1)
sns.histplot(df_a, kde=True)
plt.title('Histogram of Interval Days (Group A)')

plt.subplot(2, 2, 2)
sns.histplot(df_b, kde=True)
plt.title('Histogram of Interval Days (Group B)')

plt.subplot(2, 2, 3)
sns.histplot(df_a, kde=True)
plt.title('Q-Q Plot of Interval Days (Group A)')
plt.grid(True)

plt.subplot(2, 2, 4)
sns.histplot(df_b, kde=True)
plt.title('Q-Q Plot of Interval Days (Group B)')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Perform the Shapiro-Wilk test for normality
shapiro_a_stat, shapiro_a_p_value = shapiro(df_a)
shapiro_b_stat, shapiro_b_p_value = shapiro(df_b)

# Display the results of the Shapiro-Wilk test
print(f"Shapiro-Wilk Test for Group A: Statistic={shapiro_a_stat}, P-value={shapiro_a_p_value}")
print(f"Shapiro-Wilk Test for Group B: Statistic={shapiro_b_stat}, P-value={shapiro_b_p_value}")

Interpretation:

Histograms: The histograms show the distribution of interval days for each group. If the data follows a normal distribution, the histogram should resemble a bell curve.

Q-Q Plots: The Q-Q plots compare the quantiles of the data to the quantiles of a normal distribution. If the data follows a normal distribution, the points should lie along the diagonal line.

Shapiro-Wilk Test: The Shapiro-Wilk test checks for normality. A low p-value (typically less than 0.05) indicates that the data does not follow a normal distribution.
In this case, the p-values for both groups are extremely low, indicating that the data does not follow a normal distribution.

In [ ]:
from scipy.stats import mannwhitneyu

u_stat, p_value = mannwhitneyu(df_a.values.astype(float), df_b.values.astype(float))
print(f"U-statistic: {u_stat}, P-value: {p_value}")

print("The Mann-Whitney U test revealed a statistically significant difference in the interval "
f"days per person between Group A and Group B (U = {u_stat}, p = {p_value}), "
"indicating that the distributions of interval days differ between the two groups.")
